In [ ]:
!pip install sentencepiece
!pip uninstall -y tensorflow && pip install tensorflow-cpu

In [ ]:
import pandas as pd
import sentencepiece as spm
import json
import numpy as np
import torch
import torch.nn as nn
import math
import torch_xla.core.xla_model as xm
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import torch_xla.distributed.parallel_loader as pl
import torch.optim as optim
import os
import torch.nn.functional as F
from pathlib import Path
import torch_xla

In [ ]:
device = xm.xla_device()

In [ ]:
class Sequence_Tokenizer:
    def __init__(self, vocab_path, k=6, pad_id=0):
        self.vocab_path = vocab_path
        self.k = k
        self.pad_id = pad_id
        # Load vocab once
        with open(self.vocab_path, "r") as file:
            self.vocab = json.load(file)

    def tokenize_sequence(self, seq):
        tokens = self.kmer_tokenize(seq)
        t = [self.vocab.get(tok, self.vocab.get("<unk>", 3)) for tok in tokens]
        
        t.insert(0, 1)
        t.append(2)
        return t

    def kmer_tokenize(self, sequence):
        return [sequence[i:i+self.k] for i in range(len(sequence) - self.k + 1)]            

    def get_vocab_size(self):
        return len(self.vocab)

In [ ]:
sequence_tokenizer = Sequence_Tokenizer("/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/vocab_protein.json")
text_sp = spm.SentencePieceProcessor(model_file='/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/input_effect_text.model')
changes_sp = spm.SentencePieceProcessor(model_file='/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/changes.model')

In [ ]:
class ShardDataset(Dataset):
    def __init__(self, shard_paths):
        self.shard_paths = shard_paths
        self.current_shard = None
        self.current_data = None
        self.shard_idx = -1

        # build global index: (shard_id, local_idx)
        self.index = []
        for i, p in enumerate(self.shard_paths):
            data = torch.load(p, mmap=True)
            for j in range(len(data)):
                self.index.append((i, j))

    def __len__(self):
        return len(self.index)

    def _load_shard(self, shard_id):
        if self.shard_idx != shard_id:
            self.current_data = torch.load(
                self.shard_paths[shard_id], mmap=True
            )
            self.shard_idx = shard_id

    def __getitem__(self, idx):
        shard_id, local_idx = self.index[idx]
        self._load_shard(shard_id)
        return self.current_data[local_idx]

In [ ]:
def make_collate_fn(dna_max_len, text_max_len, out_max_len, pad_id):
    def collate(batch):
        B = len(batch)

        input_sequence = torch.full(
            (B, dna_max_len), pad_id, dtype=batch[0]["input_sequence"].dtype
        )
        input_text = torch.full(
            (B, text_max_len), pad_id, dtype=batch[0]["input_text"].dtype
        )
        output_changes = torch.full(
            (B, out_max_len), pad_id, dtype=batch[0]["output_changes"].dtype
        )

        encoder_mask = torch.ones(
            (B, dna_max_len + text_max_len), dtype=torch.bool
        )
        decoder_mask = torch.ones(
            (B, out_max_len), dtype=torch.bool
        )

        for i, b in enumerate(batch):
            l = b["input_sequence"].size(0)
            input_sequence[i, :l] = b["input_sequence"]

            l = b["input_text"].size(0)
            input_text[i, :l] = b["input_text"]

            l = b["output_changes"].size(0)
            output_changes[i, :l] = b["output_changes"]

            l = b["encoder_pad_mask"].size(0)
            encoder_mask[i, :l] = b["encoder_pad_mask"]

            l = b["decoder_pad_mask"].size(0)
            decoder_mask[i, :l] = b["decoder_pad_mask"]

        return {
            "input_sequence": input_sequence,
            "input_text": input_text,
            "output_changes": output_changes,
            "encoder_mask": encoder_mask,
            "decoder_mask": decoder_mask,
        }
    return collate

In [ ]:
BUCKET_CONFIG = {
    "1k":   {"dna": 1024,  "bs": 64},
    "4k":   {"dna": 4096,  "bs": 32},
    "16k":  {"dna": 16384, "bs": 8},
    "32k":  {"dna": 32768, "bs": 1},
    "64k":  {"dna": 65536, "bs": 1},
    "128k":  {"dna": 131072, "bs": 1},

}

def make_loader(bucket_name, root=''):
    shard_paths = sorted((Path(root) / bucket_name).glob("*.pt"))

    dataset = ShardDataset(shard_paths)

    cfg = BUCKET_CONFIG[bucket_name]
    
    try:
        # Standard for most versions
        world_size = xm.xrt_world_size()
        rank = xm.get_ordinal()
    except AttributeError:
        # Fallback for very new/specific versions
        world_size = xr.world_size()
        rank = xr.global_ordinal()

    # 1. Create the Sampler
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False
    )

    return DataLoader(
        dataset=dataset,
        batch_size=cfg["bs"],
        sampler=sampler,
        drop_last=True,
        collate_fn=make_collate_fn(
            dna_max_len=cfg["dna"],
            text_max_len=512,
            out_max_len=256,
            pad_id=0,
        ),
    )

In [ ]:
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr  # Modern runtime access

def get_tpu_loader(dataset, batch_size):
    # Try to get world_size and rank safely
    try:
        # Standard for most versions
        world_size = xm.xrt_world_size()
        rank = xm.get_ordinal()
    except AttributeError:
        # Fallback for very new/specific versions
        world_size = xr.world_size()
        rank = xr.global_ordinal()

    # 1. Create the Sampler
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True
    )

    # 2. Standard DataLoader
    make_loader()
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=make_collate_fn(),
        num_workers=4, 
        drop_last=True,
    )
    
    return loader

In [ ]:
class BaseAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, max_len, window_size=128, dropout=0.1):
        super().__init__()

        self.window_size = window_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_queries_per_kv = num_heads // num_kv_heads
        self.head_dim = d_model // num_heads
        self.d_model = d_model
        self.dropout_p = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        kv_dim = self.num_kv_heads * self.head_dim
        self.k_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Pre-compute RoPE for max possible length
        inv_freq = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer("inv_freq", inv_freq)
        
        max_possible_len = 8192 + 32  # 64k bucket + fixed input = 4128
        t = torch.arange(max_possible_len).type_as(inv_freq)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

        # ENCODER lengths (after CNN): DNA + fixed 512
        # 1k: 64 + 32 = 96
        # 4k: 256 + 32 = 288  
        # 16k: 1024 + 32 = 1056
        encoder_sizes = [96, 288, 1056, 2080, 4128, 8224]
        
        # DECODER length (no CNN compression)
        decoder_size = 255
        
        # All possible (Tq, Tk) combinations for self-attention
        all_sizes = encoder_sizes + [decoder_size]
        
        # Pre-compute masks for all combinations
        for tq in all_sizes:
            for tk in all_sizes:
                mask = self._build_sliding_mask_static(tq, tk, window_size)
                self.register_buffer(f"sliding_mask_{tq}x{tk}", mask)

    def _build_sliding_mask_static(self, tq, tk, window_size):
        """Build sliding window mask - vectorized, supports rectangular masks"""
        i = torch.arange(tq).unsqueeze(1)   # [tq, 1]
        j = torch.arange(tk).unsqueeze(0)   # [1, tk]
        dist = (i - j).abs()
        mask = dist > (window_size // 2)
        return mask  # [tq, tk], dtype=bool

    def _apply_rope(self, x, cos, sin):
        x1 = x[..., :self.head_dim // 2]
        x2 = x[..., self.head_dim // 2:]
        rotated_x = torch.cat((-x2, x1), dim=-1)
        return (x * cos) + (rotated_x * sin)

    def _get_mask(self, tq, tk):
        """Lookup pre-computed mask - no dynamic operations!"""
        # Try exact match first
        mask_name = f"sliding_mask_{tq}x{tk}"
        if hasattr(self, mask_name):
            return getattr(self, mask_name)
        
        # Fallback error
        raise ValueError(
            f"No pre-computed mask for shape ({tq}, {tk}). "
            f"Expected encoder sizes: 96, 288, 1056 or decoder size: 255"
        )

    def forward(self, q, k, v, key_padding_mask=None, attn_mask=None, is_causal=False):
        B, Tq, _ = q.shape
        Tk = k.size(1)
        
        # Get pre-computed mask - just a buffer lookup!
        window_mask = self._get_mask(Tq, Tk)
        
        # Expand for batch and heads
        window_mask = window_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, Tq, Tk]
        window_mask = window_mask.expand(B, self.num_heads, -1, -1)
    
        # Projections - ALL THREE NEED PROJECTION!
        q = self.q_proj(q).view(B, Tq, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(B, Tk, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(B, Tk, self.num_kv_heads, self.head_dim).transpose(1, 2)  # FIX: Added v_proj!
    
        # Apply RoPE with slicing (still 4 possible sizes, acceptable)
        q = self._apply_rope(q, self.cos_cached[:, :, :Tq, :], self.sin_cached[:, :, :Tq, :])
        k = self._apply_rope(k, self.cos_cached[:, :, :Tk, :], self.sin_cached[:, :, :Tk, :])
    
        # GQA broadcasting
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
    
        # Combine masks
        if key_padding_mask is not None:
            pm = key_padding_mask.view(B, 1, 1, Tk).bool()
            window_mask = window_mask | pm
        
        if attn_mask is not None:
            amask = attn_mask.view(1, 1, Tq, Tk).bool()
            window_mask = window_mask | amask
    
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=window_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False
        )
    
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.out_proj(out)

In [ ]:
class EmbeddingEncoder(nn.Module):
  def __init__(self, max_len, vocab_size_dna, vocab_size_text, d_model):
    super().__init__()
    self.d_model = d_model

    self.token_emb_dna = nn.Embedding(vocab_size_dna, d_model, padding_idx=0)
    self.token_emb_text = nn.Embedding(vocab_size_text, d_model, padding_idx=0)

    # One positional embedding table for the combined sequence
    # Segment embeddings (DNA=0, Text=1)
    self.seg_emb = nn.Embedding(2, d_model)

    self.norm = nn.LayerNorm(d_model)

  def forward(self, context_dna, context_text):
    batch_size = context_dna.size(0)
    L1 = context_dna.size(1)
    L2 = context_text.size(1)

    # Token embeddings
    dna_emb = self.token_emb_dna(context_dna)   # (B, L1, d_model)
    text_emb = self.token_emb_text(context_text) # (B, L2, d_model)

    # Concatenate
    concat_emb = torch.concat([dna_emb, text_emb], axis=1)  # (B, L1+L2, d_model)

    # Segment embeddings
    dna_seg = torch.zeros((batch_size, L1), dtype=torch.int32, device=context_dna.device)  # DNA=0
    text_seg = torch.ones((batch_size, L2), dtype=torch.int32, device=context_dna.device)  # Text=1
    seg_ids = torch.concat([dna_seg, text_seg], axis=1)      # (B, L1+L2)
    seg_emb = self.seg_emb(seg_ids)                       # (B, L1+L2, d_model)

    # Add everything
    x = concat_emb + seg_emb

    x = self.norm(x)
    
    return x

In [ ]:
class EmbeddingDecoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_model):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        self.norm = nn.LayerNorm(d_model)


    def forward(self, x):
        x = self.embedding(x)

        return self.norm(x)

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, max_len, dropout=0.1):
        super().__init__()

        self.attention = BaseAttention(d_model, num_heads, num_kv_heads, max_len=max_len, dropout=dropout)

        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, y, key_padding_mask):

        x = self.norm(x)

        y = self.norm(y)

        x_att = self.attention(x, y, y, key_padding_mask=key_padding_mask)

        return x + x_att

In [ ]:
class Self_Attention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, max_len, dropout=0.1):
        super().__init__()

        self.attention = BaseAttention(d_model, num_heads, num_kv_heads, max_len=max_len, dropout=dropout)

        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, key_padding_mask):
        x = self.norm(x)

        x_att = self.attention(x, x, x, key_padding_mask=key_padding_mask)

        return x + x_att

In [ ]:
class CausualAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, max_len, dropout=0.1):
        super().__init__()
        self.attention = BaseAttention(d_model, num_heads, num_kv_heads, max_len=max_len, dropout=dropout)
        self.norm = nn.LayerNorm(d_model)
        
        # Pre-create masks for common sequence lengths
        max_seq = 512
        full_mask = torch.triu(torch.ones(max_seq, max_seq), diagonal=1).to(torch.bool)
        self.register_buffer("causal_mask", full_mask)
        
        # Cache masks for common lengths to avoid slicing
        self.mask_cache = {}
        for length in [16, 32, 64, 128, 255, 256, 512]:  # Common lengths
            self.mask_cache[length] = full_mask[:length, :length].clone()
    
    def forward(self, x, key_padding_mask):
        seq_len = x.size(1)
        
        # Use cached mask if available, otherwise slice (but warn)
        if seq_len in self.mask_cache:
            current_mask = self.mask_cache[seq_len]
        else:
            current_mask = self.causal_mask[:seq_len, :seq_len]
        
        x = self.norm(x)
        x_att = self.attention(
            x, x, x, 
            key_padding_mask=key_padding_mask, 
            attn_mask=current_mask, 
            is_causal=True
        )
        return x + x_att

In [ ]:
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model, dff, dropout=0.1):
        super().__init__()
        # We need 2/3 of the standard dff to keep FLOPs constant
        # We multiply by 2 here because we split this layer into two (Gate & Value)
        self.w12 = nn.Linear(d_model, 2 * dff) 
        self.w3 = nn.Linear(dff, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        
        # 1. Project to 2x hidden size
        gate_plus_value = self.w12(x)
        
        # 2. Split into the Gate and the Value path
        gate, value = gate_plus_value.chunk(2, dim=-1)
        
        # 3. SwiGLU Math: SiLU(gate) * value
        # F.silu is the same as the 'Swish' activation
        x = F.silu(gate) * value
        
        # 4. Project back and add residual
        x = self.w3(x)
        return residual + self.dropout(x)

In [ ]:
class DNA_CNN_Downsampler(nn.Module):
    def __init__(
        self,
        d_model,
        kernel_sizes=[31, 15, 11, 7],
        strides=[2, 2, 2, 2],
        activation='gelu',
        dropout=0.1,
        use_batch_norm=True
    ):
        super().__init__()

        filters = [d_model // 8, d_model // 4, d_model // 2, d_model]

        self.filters = filters
        self.kernel_sizes = kernel_sizes
        self.strides = strides
        self.activation_name = activation
        self.dropout_rate = dropout
        self.use_batch_norm = use_batch_norm

        # Activation function
        if activation.lower() == 'gelu':
            self.activation = nn.GELU()
        elif activation.lower() == 'relu':
            self.activation = nn.ReLU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        # Build conv blocks
        self.conv_blocks = nn.ModuleList()
        for i, (f, k, s) in enumerate(zip(filters, kernel_sizes, strides)):
            layers = []

            # Conv1d
            layers.append(nn.Conv1d(
                in_channels=d_model if i == 0 else filters[i-1],
                out_channels=f,
                kernel_size=k,
                stride=s,
                padding=k // 2,  # 'same' padding
            ))

            # Optional BatchNorm
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(f))

            # Activation
            layers.append(self.activation)

            # Dropout
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            self.conv_blocks.append(nn.Sequential(*layers))

        # Total downsampling factor
        self.total_downsampling = 1
        for s in strides:
            self.total_downsampling *= s

        self.pool_layers = nn.ModuleList([
        nn.MaxPool1d(kernel_size=k, stride=s, padding=k//2)
        for k, s in zip(kernel_sizes, strides)
    ])


    def downsample_mask(self, mask):
        """
        mask: (batch, seq_len) 1=valid, 0=padded
        kernel_sizes: list of ints, CNN kernel sizes
        strides: list of ints, CNN strides
        """
        mask = mask.float().unsqueeze(1)  # (batch, 1, seq_len)
        for pool in self.pool_layers:
            mask = pool(mask)
            mask = (mask > 0).float()
        return mask.squeeze(1)  # (batch, seq_len_down)


    def forward(self, x, sequence_mask):
        """
        x: shape (batch, seq_len, channels)
        PyTorch Conv1d expects (batch, channels, seq_len)
        """
        # Swap to (batch, channels, seq_len)
        x = x.transpose(1, 2).contiguous()
        for block in self.conv_blocks:
            x = block(x)
        # Swap back to (batch, seq_len, channels)
        x = x.transpose(1, 2).contiguous()

        sequence_mask = self.downsample_mask(sequence_mask)

        return x, sequence_mask

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dff, num_kv_heads, max_len, dropout=0.1):
        super().__init__()

        self.attention = Self_Attention(d_model, num_heads, num_kv_heads=num_kv_heads, max_len=max_len, dropout=dropout)

        self.ffn = SwiGLUFeedForward(d_model, dff, dropout=dropout)        
    def forward(self, x, key_padding_mask):

        x = self.attention(x, key_padding_mask)

        x = self.ffn(x)

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, *, num_layers, d_model, num_heads,
                 dff, vocab_size_text, vocab_size_dna, num_kv_heads, max_len=8192, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.cnn = DNA_CNN_Downsampler(d_model=d_model, dropout=dropout)
        self.embedding = EmbeddingEncoder(max_len=max_len, 
                                                    d_model=d_model,
                                                    vocab_size_dna=vocab_size_dna, 
                                                    vocab_size_text=vocab_size_text)
        
        # FIX: Use nn.ModuleList instead of regular list
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model=d_model,
                         num_heads=num_heads,
                         dff=dff,
                         dropout=dropout,
                         num_kv_heads=num_kv_heads,
                         max_len=max_len)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)  # Also fix: should be 'dropout', not 'dropout_rate'
    def forward(self, context_dna, context_text, encoder_mask):

        x = self.embedding(context_dna, context_text)

        x, encoder_mask = self.cnn(x, encoder_mask)

        x = self.dropout(x)
        # Encoder layers
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, encoder_mask)

        return x, encoder_mask  # (B, L1+L2, d_model)

In [ ]:
class Decoder_Layer(nn.Module):
    def __init__(self, max_len, d_model, num_heads, dff, num_kv_heads, dropout=0.1):
        super().__init__()

        self.causal_attention = CausualAttention(d_model, num_heads, num_kv_heads=num_kv_heads, max_len=max_len, dropout=dropout)

        self.cross_attention = CrossAttention(d_model, num_heads, num_kv_heads=num_kv_heads, max_len=max_len, dropout=dropout)

        self.ffn = SwiGLUFeedForward(d_model, dff, dropout=dropout)   
        
    def forward(self, x, y, decoder_key_padding_mask, encoder_key_padding_mask):
        x = self.causal_attention(x, decoder_key_padding_mask)

        x = self.cross_attention(x, y, encoder_key_padding_mask)

        x = self.ffn(x)

        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, max_len, vocab_size, d_model, num_heads, dff, num_layers, num_kv_heads, dropout=0.1):
        super().__init__()

        self.embedding = EmbeddingDecoder(vocab_size, max_len, d_model)

        self.dec_layers = nn.ModuleList([
            Decoder_Layer(d_model=d_model,
                         max_len=max_len,
                         num_heads=num_heads,
                         dff=dff,
                         dropout=dropout,
                         num_kv_heads=num_kv_heads)
            for _ in range(num_layers)])
        
        self.dropout = nn.Dropout(dropout)

        self.num_layers = num_layers
        
    def forward(self, x, y, encoder_key_padding_mask, decoder_mask):
        x = self.embedding(x)
        
        x = self.dropout(x)
        
        for i in range(self.num_layers):
          x = self.dec_layers[i](x, y, decoder_mask, encoder_key_padding_mask)

        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size_text, vocab_size_dna, target_vocab_size, d_model, num_heads, 
                 dff, num_layers, max_len, num_kv_heads, dropout=0.1):
        super().__init__()

        self.encoder = Encoder(
            max_len=max_len,
            num_layers=num_layers,
            vocab_size_text=vocab_size_text, 
            vocab_size_dna=vocab_size_dna,
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            dropout=dropout,
            num_kv_heads=num_kv_heads)

        self.decoder = Decoder(
            max_len=max_len,
            num_layers=num_layers,
            vocab_size=target_vocab_size, 
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            dropout=dropout,
            num_kv_heads=num_kv_heads)
            
        self.final_linear = nn.Linear(d_model, target_vocab_size)

    def forward(self, dna, input_text, final_text, encoder_mask, decoder_mask):

        x, encoder_mask = self.encoder(dna, input_text, encoder_mask)

        x = self.decoder(final_text, x, encoder_mask, decoder_mask)

        return self.final_linear(x)

In [ ]:
num_layers = 6
d_model = 512
dff = 2048
num_heads = 8
dropout_rate = 0.2
num_kv_heads = 4
max_len = 66500

In [ ]:
model = Transformer(
    vocab_size_dna=sequence_tokenizer.get_vocab_size(),
    vocab_size_text=text_sp.get_piece_size(),
    target_vocab_size=changes_sp.get_piece_size(),
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    num_layers=num_layers,
    dropout=dropout_rate,
    max_len=max_len,
    num_kv_heads=num_kv_heads
)

In [ ]:
def lr_lambda(step, bucket="4k"):
    step = max(step + 1, 1)
    warmup_steps = 4000
    d_model = 512
    
    arg1 = step ** -0.5
    arg2 = step * (warmup_steps ** -1.5)
    base_lr = (d_model ** -0.5) * min(arg1, arg2)
    
    # Aggressive LR scaling for long sequences
    if bucket == "64k":
        return base_lr * 0.05   # 20x reduction
    elif bucket == "32k":
        return base_lr * 0.1    # 10x reduction
    elif bucket == "16k":
        return base_lr * 0.2    # 5x reduction
    elif bucket == "4k":
        return base_lr * 0.5    # 2x reduction
    else:
        return base_lr

In [ ]:
class MaskedExactMatch:
    def __init__(self, pad_id=0, device='cpu'):
        self.pad_id = pad_id
        self.device = device
        self.reset()
    
    def reset(self):
        """Reset counters as device tensors"""
        self.total_matches = torch.tensor(0, dtype=torch.long, device=self.device)
        self.total_valid = torch.tensor(0, dtype=torch.long, device=self.device)
        self.batch_accuracies = []  # Store batch-wise accuracies
    
    def compute_batch_accuracy(self, y_true, y_pred):
        """Compute accuracy for a single batch without updating internal state"""
        if y_pred.dim() > y_true.dim():
            y_pred = torch.argmax(y_pred, dim=-1)
        
        y_true = y_true.to(self.device)
        y_pred = y_pred.to(self.device)
        
        mask = (y_true != self.pad_id)
        matches = (y_true == y_pred) & mask
        
        batch_matches = matches.sum().item()
        batch_valid = mask.sum().item()
        
        if batch_valid == 0:
            return 0.0
        
        return batch_matches / batch_valid
    
    def update(self, y_true, y_pred):
        """Update metrics - keeps everything on device!"""
        if y_pred.dim() > y_true.dim():
            y_pred = torch.argmax(y_pred, dim=-1)
        
        y_true = y_true.to(self.device)
        y_pred = y_pred.to(self.device)
        
        mask = (y_true != self.pad_id)
        matches = (y_true == y_pred) & mask
        
        # Keep as tensors on device - NO .item() calls!
        self.total_matches += matches.sum()
        self.total_valid += mask.sum()
        
        # Compute and store batch accuracy
        batch_acc = self.compute_batch_accuracy(y_true, y_pred)
        self.batch_accuracies.append(batch_acc)
    
    def compute(self, reduce=True):
        """Return metric value; optionally reduce across TPU cores."""
        total_matches = self.total_matches
        total_valid = self.total_valid
        
        if reduce:
            # all_reduce expects tensors, so ensure they are tensors
            if not isinstance(total_matches, torch.Tensor):
                total_matches = torch.tensor(total_matches, device=self.device)
            if not isinstance(total_valid, torch.Tensor):
                total_valid = torch.tensor(total_valid, device=self.device)
            
            total_matches = xm.all_reduce('sum', total_matches)
            total_valid = xm.all_reduce('sum', total_valid)
        
        # Only call .item() here when explicitly computing final metric
        if isinstance(total_valid, torch.Tensor):
            total_valid_val = total_valid.item()
        else:
            total_valid_val = total_valid
            
        if total_valid_val == 0:
            return 0.0
        
        if isinstance(total_matches, torch.Tensor) and isinstance(total_valid, torch.Tensor):
            return (total_matches / total_valid).item()
        else:
            return total_matches / total_valid
    
    def get_batch_accuracies(self):
        """Get all batch accuracies and clear the list"""
        accuracies = self.batch_accuracies.copy()
        self.batch_accuracies = []
        return accuracies

In [ ]:
def train_model(model, epochs=5):
    dataset_root = '/kaggle/input/bucketed-ds/train_DS'
    device = xm.xla_device()
    model = model.to(device).to(torch.bfloat16)
    
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=1.0)
    
    # Gradient clipping per bucket
    grad_clip_norms = {
        "1k": 1.0,
        "4k": 0.5,
        "16k": 0.3,
        "32k": 0.2,
        "64k": 0.1
    }
    
    LOG_EVERY = 300
    
    optimizers = {}
    schedulers = {}
    
    for bucket in ["1k", "4k", "16k", "32k", "64k"]:
        optimizers[bucket] = optim.Adam(model.parameters(), lr=1)
        
        def make_lr_lambda(bucket_name):
            def lr_fn(step):
                return lr_lambda(step, bucket=bucket_name)
            return lr_fn
        
        schedulers[bucket] = optim.lr_scheduler.LambdaLR(
            optimizers[bucket], 
            make_lr_lambda(bucket)
        )

    for epoch in range(epochs):
        # Create global and interval metrics
        global_metric = MaskedExactMatch(pad_id=0, device=device)
        interval_metric = MaskedExactMatch(pad_id=0, device=device)
        
        model.train()
        
        # Training schedule
        training_schedule = [
            ("1k", 9),
            ("4k", 60),      # Split into two
            ("16k", 140),    # Split into two
            ("32k", 18),
            ("4k", 57),
            ("16k", 144),
            ("64k", 5),
        ]
        
        # Track stats
        bucket_steps = {b: 0 for b in ["1k", "4k", "16k", "32k", "64k"]}
        bucket_loss_accum = {b: 0.0 for b in ["1k", "4k", "16k", "32k", "64k"]}
        bucket_step_count = {b: 0 for b in ["1k", "4k", "16k", "32k", "64k"]}
        
        cpu_losses = {b: [] for b in ["1k", "4k", "16k", "32k", "64k"]}
        cpu_accuracies = {b: [] for b in ["1k", "4k", "16k", "32k", "64k"]}
        current_bucket_losses = {b: [] for b in ["1k", "4k", "16k", "32k", "64k"]}
        current_bucket_accuracies = {b: [] for b in ["1k", "4k", "16k", "32k", "64k"]}
        
        # Prepare all iterators
        loaders = {}
        iterators = {}
        for bucket in ["1k", "4k", "16k", "32k", "64k"]:
            loaders[bucket] = make_loader(bucket, dataset_root)
            tpu_loader = pl.ParallelLoader(loaders[bucket], [device]).per_device_loader(device)
            iterators[bucket] = iter(tpu_loader)
        
        exhausted_buckets = set()
        schedule_idx = 0
        global_step = 0
        
        while len(exhausted_buckets) < 5:
            bucket, num_steps = training_schedule[schedule_idx % len(training_schedule)]
            schedule_idx += 1
            
            if bucket in exhausted_buckets:
                continue
            
            max_grad_norm = grad_clip_norms[bucket]
            optimizer = optimizers[bucket]
            scheduler = schedulers[bucket]
            
            # Train for num_steps on this bucket
            for _ in range(num_steps):
                try:
                    batch = next(iterators[bucket])
                except StopIteration:
                    exhausted_buckets.add(bucket)
                    xm.master_print(f"Bucket {bucket} exhausted after {bucket_steps[bucket]} steps")
                    break
                
                optimizer.zero_grad()
                
                # Prepare inputs
                encoder_seq = batch['input_sequence'].to(device)
                encoder_text = batch['input_text'].to(device)
                target = batch['output_changes'].to(device)
                encoder_mask = batch['encoder_mask'].to(device)
                decoder_mask = batch['decoder_mask'].to(device)
                
                decoder_mask = decoder_mask[:, :-1]
                decoder_input = target[:, :-1]
                decoder_target = target[:, 1:]
                
                # Forward pass
                outputs = model(encoder_seq, encoder_text, decoder_input, encoder_mask, decoder_mask)
                loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_target.view(-1))
                
                # Backward pass
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                xm.optimizer_step(optimizer)
                scheduler.step()
                
                # Force computation and get loss value
                loss_value = loss.detach()
                xm.mark_step()
                
                # Compute batch accuracy
                batch_acc = interval_metric.compute_batch_accuracy(decoder_target, outputs)
                
                # Update metrics
                global_metric.update(decoder_target, outputs)
                interval_metric.update(decoder_target, outputs)
                
                # Update counters and storage
                bucket_steps[bucket] += 1
                global_step += 1
                
                current_bucket_losses[bucket].append(loss_value.cpu().item())
                current_bucket_accuracies[bucket].append(batch_acc)
                bucket_loss_accum[bucket] += loss_value.cpu().item()
                bucket_step_count[bucket] += 1
                
                # Logging every LOG_EVERY steps
                if bucket_steps[bucket] % LOG_EVERY == 0:
                    # Calculate average loss and accuracy for this interval
                    avg_loss = bucket_loss_accum[bucket] / bucket_step_count[bucket]
                    
                    # Get interval accuracy (average of batch accuracies)
                    interval_accs = interval_metric.get_batch_accuracies()
                    avg_interval_acc = sum(interval_accs) / len(interval_accs) if interval_accs else 0.0
                    
                    # Update learning rate
                    current_lr = lr_lambda(global_step, bucket=bucket)
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = current_lr
                    
                    # Log with both loss and accuracy
                    xm.master_print(
                        f"Epoch {epoch} | Bucket {bucket} | Step {bucket_steps[bucket]} | "
                        f"Loss {avg_loss:.4f} | Acc {avg_interval_acc:.4f} | LR {current_lr:.6f}"
                    )
                    
                    # Save accumulated losses and accuracies
                    cpu_losses[bucket].extend(current_bucket_losses[bucket])
                    cpu_accuracies[bucket].extend(current_bucket_accuracies[bucket])
                    current_bucket_losses[bucket] = []
                    current_bucket_accuracies[bucket] = []
                    
                    # Reset accumulators
                    bucket_loss_accum[bucket] = 0.0
                    bucket_step_count[bucket] = 0
        
        # Final save for any remaining losses and accuracies
        for bucket in ["1k", "4k", "16k", "32k", "64k"]:
            if current_bucket_losses[bucket]:
                cpu_losses[bucket].extend(current_bucket_losses[bucket])
                cpu_accuracies[bucket].extend(current_bucket_accuracies[bucket])
        
        # Compute global metric for the epoch
        xm.mark_step()
        global_acc = global_metric.compute(reduce=True)
        if torch.is_tensor(global_acc):
            global_acc = global_acc.cpu().item()
        
        xm.master_print(f"--- Epoch {epoch} completed | Train Masked Exact Match: {global_acc:.4f} ---")
        
        # Save losses and accuracies
        if xm.is_master_ordinal():
            torch.save(cpu_losses, f"train_losses_2_epoch_{epoch}.pt")
            torch.save(cpu_accuracies, f"train_accuracies_2_epoch_{epoch}.pt")
    
    return model

In [ ]:
epochs=1

In [ ]:
train_model(model, epochs=epochs)

In [ ]:
def evaluate(model, device, criterion, checkpoint_path=None, pad_id=0):
    import torch_xla.core.xla_model as xm
    
    if checkpoint_path is not None:
        xm.master_print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        xm.master_print(f"Loaded model from epoch {checkpoint['epoch']}")
    
    dataset_root = '/kaggle/input/bucketed-ds/test_DS'
    model = model.to(device).to(torch.bfloat16)
    
    model.eval()
    metric = MaskedExactMatch(pad_id=pad_id, device=device)
    
    LOG_EVERY = 250
    buckets = ["1k", "4k", "16k", "32k", "64k"]
    
    # Initialize accumulators
    total_loss = torch.tensor(0.0, device=device, dtype=torch.bfloat16)
    total_steps = torch.tensor(0, device=device, dtype=torch.int32)
    
    cpu_losses = {b: [] for b in buckets}
    cpu_accuracies = {b: [] for b in buckets}
    current_bucket_losses = {b: [] for b in buckets}
    current_bucket_accuracies = {b: [] for b in buckets}
    
    with torch.no_grad():
        for bucket in buckets:
            loader = make_loader(bucket, dataset_root)
            tpu_loader = pl.ParallelLoader(loader, [device]).per_device_loader(device)
            
            bucket_loss = torch.tensor(0.0, device=device)
            bucket_steps = 0
            
            for batch_idx, batch in enumerate(tpu_loader):
                # Prepare inputs
                encoder_seq = batch['input_sequence'].to(device)
                encoder_text = batch['input_text'].to(device)
                target = batch['output_changes'].to(device)
                encoder_mask = batch['encoder_mask'].to(device)
                decoder_mask = batch['decoder_mask'].to(device)
                
                decoder_mask = decoder_mask[:, :-1]
                decoder_input = target[:, :-1]
                decoder_target = target[:, 1:]
                
                # Forward pass
                outputs = model(encoder_seq, encoder_text, decoder_input, encoder_mask, decoder_mask)
                loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_target.view(-1))
                
                # Compute batch accuracy
                batch_acc = metric.compute_batch_accuracy(decoder_target, outputs)
                
                # Update metric
                metric.update(decoder_target, outputs)
                
                # Accumulate
                total_loss += loss.detach()
                total_steps += 1
                bucket_loss += loss.detach()
                bucket_steps += 1
                
                # Store batch results
                current_bucket_losses[bucket].append(loss.detach().cpu().item())
                current_bucket_accuracies[bucket].append(batch_acc)
                
                # Log progress periodically
                if (batch_idx + 1) % LOG_EVERY == 0 or batch_idx == len(loader) - 1:
                    xm.mark_step()
                    
                    # Log current bucket progress
                    avg_batch_loss = bucket_loss.item() / bucket_steps
                    batch_accs = current_bucket_accuracies[bucket][-LOG_EVERY:] if len(current_bucket_accuracies[bucket]) > LOG_EVERY else current_bucket_accuracies[bucket]
                    avg_batch_acc = sum(batch_accs) / len(batch_accs) if batch_accs else 0.0
                    
                    xm.master_print(
                        f"Eval | Bucket {bucket} | Batch {batch_idx+1}/{len(loader)} | "
                        f"Loss {avg_batch_loss:.4f} | Acc {avg_batch_acc:.4f}"
                    )
            
            xm.master_print(f"Bucket: {bucket} finished")
            
            # Save bucket results
            cpu_losses[bucket].extend(current_bucket_losses[bucket])
            cpu_accuracies[bucket].extend(current_bucket_accuracies[bucket])
    
    # Wait for all computations to complete
    xm.mark_step()
    
    # Compute final metrics
    avg_loss_tensor = total_loss / total_steps
    avg_loss_tensor = xm.all_reduce('sum', avg_loss_tensor) / xr.world_size()
    avg_loss = avg_loss_tensor.cpu().item()
    
    masked_acc_tensor = metric.compute(reduce=True)
    masked_acc = masked_acc_tensor.cpu().item() if torch.is_tensor(masked_acc_tensor) else masked_acc_tensor
    
    xm.master_print(f"Evaluation complete: Loss={avg_loss:.4f}, Accuracy={masked_acc:.4f}")
    
    # Save results
    if xm.is_master_ordinal():
        torch.save(cpu_losses, "test_losses_2.pt")
        torch.save(cpu_accuracies, "test_accuracies_2.pt")
    
    return {
        'loss': avg_loss, 
        'masked_exact_match': masked_acc,
        'per_bucket_losses': cpu_losses,
        'per_bucket_accuracies': cpu_accuracies
    }

In [ ]:
device = xm.xla_device()
criterion = nn.CrossEntropyLoss(ignore_index=0)

result = evaluate(model, device, criterion)
xm.master_print(result)